In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml kailash-kaizen

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# M6 Ollama bootstrap — local LLMs, no API keys
# On Colab: installs + starts Ollama and pulls this lesson's models.
# Locally: verifies the daemon is running and models are pulled.
# ══════════════════════════════════════════════════════════════════
_LESSON_MODELS = ['llama3.2:3b']
import sys, os, time, shutil, subprocess
import httpx

_OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
_IN_COLAB = 'google.colab' in sys.modules

def _daemon_up(timeout_s=1.0):
    try:
        r = httpx.get(f'{_OLLAMA_BASE_URL}/api/tags', timeout=timeout_s)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

if _IN_COLAB and shutil.which('ollama') is None:
    print('[ollama] installing binary …')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )

if _IN_COLAB and _daemon_up() is None:
    print('[ollama] starting daemon …')
    subprocess.Popen(
        'nohup ollama serve > /tmp/ollama.log 2>&1 &',
        shell=True,
    )
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and _daemon_up() is None:
        time.sleep(0.5)

_tags = _daemon_up(timeout_s=2.0)
if _tags is None:
    raise RuntimeError(
        f'Ollama daemon not reachable at {_OLLAMA_BASE_URL}. '
        'On Colab: re-run this cell. Locally: run `ollama serve`.'
    )

_have = {m.get('name','').split(':',1)[0] for m in _tags.get('models', [])}
for _model in _LESSON_MODELS:
    _fam = _model.split(':', 1)[0]
    if _fam in _have:
        continue
    if _IN_COLAB:
        print(f'[ollama] pulling {_model} (one-time) …')
        subprocess.run(['ollama', 'pull', _model], check=True)
    else:
        raise RuntimeError(
            f'Model {_model!r} not pulled. Run: ollama pull {_model}'
        )

print(
    '✓ Ollama ready at', _OLLAMA_BASE_URL,
    '— models:', _LESSON_MODELS,
)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp06
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


def create_visualizer():
    """Return a ModelVisualizer with the P2 experimental notice acknowledged.

    kailash-ml 2.2.2 emits ExperimentalWarning (a UserWarning) at
    ModelVisualizer construction; exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly here — the ONE construction site
    every course file should use.
    """
    import warnings as _warnings

    from kailash_ml import ModelVisualizer
    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()

# ── SDK-internal deprecation nag filter (strict-gate hygiene) ────────────
# torch 2.12's ONNX exporter warns that dynamic_axes is "not recommended"
# under dynamo=True — a deprecation notice inside OnnxBridge's internals,
# not a correctness signal, and fatal under warnings-as-errors. Acknowledged
# here (kailash_helpers is imported by every shared.mlfpNN package, so every
# technique file gets it); message-matched, everything else still errors.
import warnings as _warnings

_warnings.filterwarnings(
    "ignore", message=r".*dynamic_axes.*not recommended.*", category=UserWarning
)
# matplotlib's Agg backend (forced in headless/script runs) nags
# "FigureCanvasAgg is non-interactive, and thus cannot be shown" on every
# plt.show() — a backend notice, not a correctness signal.
_warnings.filterwarnings(
    "ignore", message=r".*non-interactive.*cannot be shown.*", category=UserWarning
)

def hdb_storey_range_expr(column: str = "storey_range"):
    """Polars expression normalising HDB storey_range letter-O typos.

    The raw HDB resale file has letter-O typos ("O4 TO 06", "1O TO 12",
    "28 TO 3O"); read a letter O next to a digit as zero, leaving "TO"
    alone. Single home for the rule (P2): M2 ex_8 and M3 ex_5/07 both
    normalise at the use-site.
    """
    import polars as pl

    return (
        pl.col(column)
        .str.replace_all(r"\bO(\d)", "0${1}")
        .str.replace_all(r"(\d)O\b", "${1}0")
    )



# ── shared/mlfp06/_ollama_bootstrap.py ──
"""Ollama bootstrap for MLFP06 — local LLMs, no API keys, real calls only.

Why this module exists
----------------------
M6 trains, prompts, evaluates, and deploys real LLMs. Routing every call
through OpenAI:

  * forces every learner to hold an API key
  * burns budget every time a notebook reruns a cell
  * makes "did the LLM actually fire?" invisible (a stubbed `cost=$0`
    looks identical to a successful call)

This module switches every M6 LLM call to a locally-running Ollama daemon
that the student installs once and re-uses across every lesson.

Contract
--------
* :func:`make_delegate` — the only sanctioned way to construct a Kaizen
  ``Delegate`` for M6. Wraps the Ollama adapter with sensible defaults and
  forces ``budget_usd=None`` (Kaizen mis-prices Ollama as $3/$15-per-Mtok;
  budget tracking against a free local provider is meaningless).
* :func:`make_embedder` — the only sanctioned way to construct an embedding
  adapter for M6 RAG (lesson 6.4).
* :func:`preflight_ollama` — pre-flight reachability + model-presence check.
  Raises :class:`OllamaUnreachableError` with an actionable message. Call
  this at the top of every notebook / script before constructing a
  Delegate.
* :data:`LESSON_MODELS` — manifest of which models each lesson needs.
  Consumed by ``scripts/generate_selfcontained_notebook.py`` to inject the
  Colab bootstrap cell.

Per ``rules/zero-tolerance.md`` Rule 2: there is no silent stub fallback
in this module. If Ollama is not reachable, we raise. The previous
``run_delegate(...) -> ("unknown", 0.0, latency)`` fallback is BLOCKED.

Per ``rules/framework-first.md``: every LLM call goes through Kaizen
``Delegate``. Raw ``httpx`` calls to ``/api/generate`` are BLOCKED for
generation paths; ``httpx`` is used here only for the daemon-reachability
preflight.
"""

import logging
import os
import shutil
import subprocess
import sys
import time
from typing import Any

import httpx
from dotenv import load_dotenv

logger = logging.getLogger(__name__)

# ════════════════════════════════════════════════════════════════════════
# Configuration — env-driven per rules/env-models.md
# ════════════════════════════════════════════════════════════════════════

load_dotenv()


def _env(key: str, default: str) -> str:
    val = os.environ.get(key)
    return val if val else default


OLLAMA_BASE_URL: str = _env("OLLAMA_BASE_URL", "http://localhost:11434")
"""Where the Ollama daemon listens. Local default; Colab uses the same."""

DEFAULT_CHAT_MODEL: str = _env("OLLAMA_CHAT_MODEL", "llama3.2:3b")
"""Tool-capable instruct model. Allowlisted by Kaizen for function calling."""

DEFAULT_EMBED_MODEL: str = _env("OLLAMA_EMBED_MODEL", "nomic-embed-text")
"""Compact embedding model — ~270MB, 768-dim. Used by M6 RAG (lesson 6.4)."""

DEFAULT_FT_BASE_MODEL: str = _env("OLLAMA_FT_BASE_MODEL", "qwen2.5:0.5b")
"""Tiny base model for fine-tuning demos (lessons 6.2, 6.3). Fits T4."""


# ════════════════════════════════════════════════════════════════════════
# Lesson → models manifest (used by generator + preflight)
# ════════════════════════════════════════════════════════════════════════


LESSON_MODELS: dict[str, list[str]] = {
    # 6.1 Prompting — chat only
    "ex_1": [DEFAULT_CHAT_MODEL],
    # 6.2 Fine-tuning — base for LoRA + chat for evaluation
    "ex_2": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.3 Alignment (DPO/GRPO) — base + chat for judge
    "ex_3": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.4 RAG — chat + embeddings
    "ex_4": [DEFAULT_CHAT_MODEL, DEFAULT_EMBED_MODEL],
    # 6.5 Agents (ReAct) — tool-capable chat
    "ex_5": [DEFAULT_CHAT_MODEL],
    # 6.6 Multi-agent + MCP — tool-capable chat
    "ex_6": [DEFAULT_CHAT_MODEL],
    # 6.7 PACT governance — chat for governed Delegate
    "ex_7": [DEFAULT_CHAT_MODEL],
    # 6.8 Nexus deployment — chat behind the API
    "ex_8": [DEFAULT_CHAT_MODEL],
}


# ════════════════════════════════════════════════════════════════════════
# Errors
# ════════════════════════════════════════════════════════════════════════


class OllamaUnreachableError(RuntimeError):
    """Daemon not running, daemon unreachable, or required model not pulled.

    The error message MUST be actionable: it tells the user the exact
    command to run to fix the situation. We never wrap a low-level
    ``httpx.ConnectError`` and re-raise without translating it.
    """


# ════════════════════════════════════════════════════════════════════════
# Preflight — reachability + model presence
# ════════════════════════════════════════════════════════════════════════


def _running_in_colab() -> bool:
    return "google.colab" in sys.modules


def _list_pulled_families(*, timeout_s: float = 2.0) -> set[str]:
    """Hit /api/tags and return the set of pulled model families (no tag).

    Returns ``{"llama3.2", "qwen2.5"}`` for a daemon with
    ``llama3.2:3b`` and ``qwen2.5:0.5b`` pulled. The model family is
    everything before the first colon — we match against family rather
    than the full ``family:tag`` because pulling ``llama3.2:3b-instruct``
    or ``llama3.2:3b-instruct-q4_K_M`` should still satisfy a request
    for the ``llama3.2`` family.
    """
    url = f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags"
    try:
        r = httpx.get(url, timeout=timeout_s)
        r.raise_for_status()
    except (httpx.HTTPError, httpx.ConnectError) as exc:
        raise OllamaUnreachableError(
            f"ollama daemon not reachable at {OLLAMA_BASE_URL}. "
            f"Fix: open a terminal and run `ollama serve`. "
            f"On macOS the desktop app starts the daemon automatically. "
            f"Underlying: {type(exc).__name__}: {exc}"
        ) from exc
    payload = r.json() or {}
    families: set[str] = set()
    for entry in payload.get("models", []):
        name = entry.get("name") or entry.get("model") or ""
        if not name:
            continue
        families.add(name.split(":", 1)[0])
    return families


def preflight_ollama(
    *,
    required_models: list[str] | None = None,
    auto_pull: bool | None = None,
    timeout_s: float = 2.0,
) -> None:
    """Verify the daemon is reachable and the required models are pulled.

    Args:
        required_models: Models that must be present, e.g.
            ``["llama3.2:3b", "nomic-embed-text"]``. Family-matched (the
            ``:tag`` portion is informational; any pulled tag for the
            same family satisfies the requirement).
        auto_pull: If ``True``, attempt ``ollama pull <model>`` for any
            missing model via subprocess. Defaults to ``True`` on Colab
            (where the bootstrap cell has just installed the binary)
            and ``False`` locally (where pulling is the student's job
            and surprise downloads are unwelcome).
        timeout_s: HTTP timeout for the reachability check. The default
            is intentionally short — if the daemon is alive it answers
            within 100ms.

    Raises:
        OllamaUnreachableError: Daemon unreachable, or a required model
            is missing after pull (or auto_pull was disabled).
    """
    if auto_pull is None:
        auto_pull = _running_in_colab()

    have = _list_pulled_families(timeout_s=timeout_s)
    needed = required_models or []
    missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing and auto_pull:
        for model in missing:
            _pull_model(model)
        # Re-check
        have = _list_pulled_families(timeout_s=timeout_s)
        missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing:
        cmds = "\n  ".join(f"ollama pull {m}" for m in missing)
        raise OllamaUnreachableError(
            f"Required model(s) not pulled: {missing}.\n"
            f"Fix — run in a terminal:\n  {cmds}"
        )

    logger.info(
        "ollama.preflight.ok",
        extra={
            "base_url": OLLAMA_BASE_URL,
            "models_present": sorted(have),
            "required": needed,
        },
    )


def _pull_model(model: str) -> None:
    """Run ``ollama pull <model>`` as a subprocess. Streams to stdout."""
    if shutil.which("ollama") is None:
        raise OllamaUnreachableError(
            f"`ollama` binary not on PATH — cannot pull {model}. "
            f"Install: https://ollama.com/download"
        )
    print(f"[ollama] pulling {model} (one-time download) …", flush=True)
    t0 = time.monotonic()
    proc = subprocess.run(
        ["ollama", "pull", model],
        capture_output=False,
        text=True,
    )
    elapsed = time.monotonic() - t0
    if proc.returncode != 0:
        raise OllamaUnreachableError(
            f"`ollama pull {model}` failed with exit {proc.returncode} "
            f"after {elapsed:.1f}s. Check network and disk space."
        )
    print(f"[ollama] {model} pulled in {elapsed:.1f}s", flush=True)


# ════════════════════════════════════════════════════════════════════════
# Delegate factory — the only sanctioned constructor for M6
# ════════════════════════════════════════════════════════════════════════


def make_delegate(
    *,
    model: str | None = None,
    tools: Any = None,
    system_prompt: str | None = None,
    temperature: float = 0.4,
    max_tokens: int = 4096,
    **kwargs: Any,
) -> Any:
    """Construct a Kaizen ``Delegate`` backed by the local Ollama daemon.

    This is the ONLY way M6 helpers and exercises construct a Delegate.
    Direct ``Delegate(...)`` construction in M6 code is BLOCKED — the
    cost-mispricing trap (``budget_usd`` against a free provider) and
    the silent OpenAI fallback that comes from omitting ``adapter=`` are
    the reasons.

    Args:
        model: Ollama model tag (e.g. ``"llama3.2:3b"``). Defaults to
            :data:`DEFAULT_CHAT_MODEL`.
        tools: Tool registry or list — passed through to Delegate.
            Auto-disables streaming on the Ollama adapter.
        system_prompt: Optional system prompt.
        temperature: Sampling temperature. Default 0.4.
        max_tokens: Hard cap on output length. Default 4096.
        **kwargs: Forwarded to ``Delegate(...)``. ``budget_usd`` is
            forced to ``None`` and ``api_key`` / ``base_url`` are
            BLOCKED (the adapter owns those).

    Returns:
        A configured Kaizen ``Delegate`` instance.
    """
    for blocked in ("api_key", "base_url", "budget_usd", "adapter"):
        if blocked in kwargs:
            raise ValueError(
                f"make_delegate(): {blocked!r} is owned by the bootstrap "
                f"and cannot be overridden by callers."
            )

    # Lazy imports — keep this module importable in environments that have
    # not yet installed kaizen-agents (e.g. early bootstrap cells).
    from kaizen_agents import Delegate
    from kaizen_agents.delegate.adapters.registry import get_adapter

    resolved = model or DEFAULT_CHAT_MODEL
    adapter = get_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return Delegate(
        model=resolved,
        adapter=adapter,
        tools=tools,
        system_prompt=system_prompt,
        budget_usd=None,
        **kwargs,
    )


# ════════════════════════════════════════════════════════════════════════
# Embedding factory — the only sanctioned constructor for M6 RAG
# ════════════════════════════════════════════════════════════════════════


def make_embedder(*, model: str | None = None) -> Any:
    """Construct a Kaizen embedding adapter backed by Ollama.

    Args:
        model: Ollama embedding model tag. Defaults to
            :data:`DEFAULT_EMBED_MODEL`.

    Returns:
        An adapter with ``async def embed(texts: list[str]) -> list[list[float]]``.
    """
    from kaizen_agents.delegate.adapters.registry import get_embedding_adapter

    resolved = model or DEFAULT_EMBED_MODEL
    return get_embedding_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
    )


# ════════════════════════════════════════════════════════════════════════
# Streaming helpers — same call shape as the old `run_delegate`
# ════════════════════════════════════════════════════════════════════════


async def run_delegate_text(
    delegate: Any, prompt: str
) -> tuple[str, dict[str, int], float]:
    """Run a Delegate to completion, return ``(text, usage, latency_s)``.

    ``usage`` is a dict with ``prompt_tokens``, ``completion_tokens``,
    ``total_tokens`` (zero for events that don't carry usage). ``cost``
    is intentionally absent — Ollama is free, and Kaizen's ``consumed_usd``
    is mis-priced for local models.

    Event handling: the Kaizen ``TextDelta`` events expose the streamed
    chunk on ``.text`` (not ``.delta_text`` — that name is reserved for a
    different StreamEvent shape). The terminal ``TurnComplete`` event
    carries the final ``.text`` (entire response, not a delta) plus the
    ``.usage`` dict. We accumulate from ``TextDelta.text`` for streaming
    UX and fall back to ``TurnComplete.text`` if no deltas arrived.
    """
    t0 = time.perf_counter()
    response = ""
    usage: dict[str, int] = {
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
    }
    final_text: str | None = None
    async for event in delegate.run(prompt):
        ev_type = getattr(event, "event_type", None)
        text = getattr(event, "text", None)
        if ev_type == "text_delta" and text:
            response += text
        elif ev_type == "turn_complete":
            final_text = text or final_text
            ev_usage = getattr(event, "usage", None) or {}
            for k in usage:
                if k in ev_usage:
                    usage[k] = int(ev_usage[k])
    if not response and final_text:
        response = final_text
    elapsed = time.perf_counter() - t0
    return response, usage, elapsed


# ════════════════════════════════════════════════════════════════════════
# Colab installer — used by the generator's Cell 0
# ════════════════════════════════════════════════════════════════════════


def colab_install_and_start_ollama(*, log_path: str = "/tmp/ollama.log") -> None:
    """Install the Ollama binary and start the daemon (Colab only).

    Idempotent: skips install if the binary is already on PATH and
    skips daemon start if a process is already listening on
    :data:`OLLAMA_BASE_URL`. Raises :class:`OllamaUnreachableError`
    if either step fails.

    Not for local use — locally, students run ``ollama serve`` themselves.
    """
    if not _running_in_colab():
        raise RuntimeError(
            "colab_install_and_start_ollama() is Colab-only. "
            "On a local machine, run `ollama serve` in a terminal."
        )

    # Step 1 — install if missing
    if shutil.which("ollama") is None:
        print("[colab] installing ollama …", flush=True)
        proc = subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True,
            check=False,
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            raise OllamaUnreachableError(
                f"ollama install failed (exit {proc.returncode}):\n"
                f"stdout: {proc.stdout[-500:]}\nstderr: {proc.stderr[-500:]}"
            )

    # Step 2 — start daemon if not already listening
    try:
        httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
        print("[colab] ollama daemon already running", flush=True)
        return
    except (httpx.HTTPError, httpx.ConnectError):
        pass

    print(f"[colab] starting ollama daemon (logs → {log_path}) …", flush=True)
    subprocess.Popen(
        f"nohup ollama serve > {log_path} 2>&1 &",
        shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # Step 3 — wait up to 30s for the daemon to answer /api/tags
    deadline = time.monotonic() + 30.0
    last_err: Exception | None = None
    while time.monotonic() < deadline:
        try:
            httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
            print("[colab] ollama daemon ready", flush=True)
            return
        except (httpx.HTTPError, httpx.ConnectError) as exc:
            last_err = exc
            time.sleep(0.5)

    raise OllamaUnreachableError(
        f"ollama daemon failed to come up within 30s. "
        f"Last error: {type(last_err).__name__}: {last_err}. "
        f"Inspect {log_path} for details."
    )


# ── shared/mlfp06/ex_5.py ──
"""
Shared infrastructure for MLFP06 Exercise 5 — AI Agents (ReAct, Structured,
Critic, Cost-Bounded).

Contains:
  - HotpotQA multi-hop QA dataset loading (cached parquet)
  - Agent tools: data_summary, search_documents, run_query, answer_question
  - build_tool_registry(): wraps the Python tools in a Kaizen ToolRegistry
    so an Ollama-backed Delegate can actually call them
  - require_llm_trace() / require_agent_result(): turn a failed LLM call
    into a loud, actionable error instead of an empty "result"
  - Model resolution from environment
  - Output directory setup

Technique-specific agent classes and signatures live in the per-technique
files under modules/mlfp06/solutions/ex_5/.
"""

import inspect
import json
import re
from pathlib import Path
from typing import Any, Callable

import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()


MODEL = DEFAULT_CHAT_MODEL

OLLAMA_HINT = (
    "Is the local Ollama daemon running? Start it with `ollama serve` and "
    f"pull the chat model with `ollama pull {MODEL}`."
)

OUTPUT_DIR = Path("outputs") / "ex5_agents"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATASET — HotpotQA (multi-hop QA, 500 cached examples)
# ════════════════════════════════════════════════════════════════════════

CACHE_DIR = Path("data") / "mlfp06" / "hotpotqa"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / "hotpotqa_500.parquet"


def load_hotpotqa() -> pl.DataFrame:
    """Load HotpotQA distractor split (500 shuffled examples, cached).

    Returns:
        Polars DataFrame with columns: text, question, answer, level, type.
    """
    if CACHE_FILE.exists():
        print(f"Loading cached HotpotQA from {CACHE_FILE}")
        return pl.read_parquet(CACHE_FILE)

    print("Downloading hotpotqa/hotpot_qa from HuggingFace...")
    from datasets import load_dataset

    ds = load_dataset(
        "hotpotqa/hotpot_qa",
        "distractor",
        split="validation",
    )
    ds = ds.shuffle(seed=42).select(range(min(500, len(ds))))
    rows = []
    for row in ds:
        context = row["context"]
        titles = context["title"]
        sentences = context["sentences"]
        joined = "\n".join(f"[{t}] " + " ".join(s) for t, s in zip(titles, sentences))
        rows.append(
            {
                "text": joined[:4000],
                "question": row["question"],
                "answer": row["answer"],
                "level": row["level"],
                "type": row["type"],
            }
        )
    qa_data = pl.DataFrame(rows)
    qa_data.write_parquet(CACHE_FILE)
    print(f"Cached {qa_data.height} HotpotQA examples at {CACHE_FILE}")
    return qa_data


# ════════════════════════════════════════════════════════════════════════
# AGENT TOOLS — docstrings ARE the agent's API documentation
# ════════════════════════════════════════════════════════════════════════
#
# Tool docstrings are what the LLM reads to choose WHICH tool to call and
# WHAT arguments to pass. Precise docstrings -> accurate tool selection.
# The tools close over a module-level `_qa_data` DataFrame populated by
# `make_tools()`.

_qa_data: pl.DataFrame | None = None


def _require_data() -> pl.DataFrame:
    if _qa_data is None:
        raise RuntimeError(
            "Agent tools used before make_tools() was called — "
            "call shared.mlfp06.ex_5.make_tools(qa_data) first."
        )
    return _qa_data


def data_summary(dataset_name: str = "qa_data") -> str:
    """Get a statistical summary of the QA dataset.

    Args:
        dataset_name: Which dataset to summarise.  Currently 'qa_data'.

    Returns:
        Text summary including shape, columns, type distribution, and
        average text lengths.
    """
    df = _require_data()
    parts = [
        f"Dataset: {dataset_name}",
        f"Shape: {df.height} rows x {df.width} columns",
        f"Columns: {', '.join(df.columns)}",
    ]
    for col in df.columns:
        dtype = str(df.schema[col])
        if "Utf8" in dtype or "String" in dtype:
            n_unique = df.select(pl.col(col).n_unique()).item()
            avg_len = df.select(pl.col(col).str.len_chars().mean()).item()
            parts.append(f"  {col} ({dtype}): {n_unique} unique, avg_len={avg_len:.0f}")
        elif "Int" in dtype or "Float" in dtype:
            stats = df.select(
                pl.col(col).mean().alias("mean"),
                pl.col(col).min().alias("min"),
                pl.col(col).max().alias("max"),
            ).row(0)
            parts.append(
                f"  {col} ({dtype}): mean={stats[0]}, range=[{stats[1]}, {stats[2]}]"
            )
    return "\n".join(parts)


def search_documents(query: str, top_k: int = 3) -> str:
    """Search the QA corpus for documents matching a keyword query.

    Args:
        query: Keywords to search for in the document texts.
        top_k:  Maximum number of matching documents to return.

    Returns:
        Matching document excerpts with their questions and answers.
    """
    df = _require_data()
    query_lower = query.lower()
    scored = []
    for i, row in enumerate(df.iter_rows(named=True)):
        text = row["text"].lower()
        score = sum(1 for word in query_lower.split() if word in text)
        if score > 0:
            scored.append((score, i, row))
    scored.sort(key=lambda x: x[0], reverse=True)

    results = []
    for score, idx, row in scored[:top_k]:
        results.append(
            f"[Doc {idx}] Q: {row['question']}\n"
            f"  A: {row['answer']}\n"
            f"  Context (excerpt): {row['text'][:300]}..."
        )
    return "\n\n".join(results) if results else f"No documents matching '{query}'"


def run_query(query_description: str) -> str:
    """Run a descriptive query against the QA dataset.

    Args:
        query_description: Natural language description of the query
            (e.g., 'count comparison questions', 'find bridge-type questions').

    Returns:
        Query results as formatted text.
    """
    df = _require_data()
    desc = query_description.lower()

    if "count" in desc and "type" in desc:
        counts = df.group_by("type").len().sort("len", descending=True)
        return f"Question types:\n{counts}"
    elif "count" in desc and "level" in desc:
        counts = df.group_by("level").len().sort("len", descending=True)
        return f"Difficulty levels:\n{counts}"
    elif "comparison" in desc:
        comparison = df.filter(pl.col("type") == "comparison")
        return (
            f"Comparison questions: {comparison.height}\n"
            f"Sample: {comparison['question'][0]}"
        )
    elif "bridge" in desc:
        bridge = df.filter(pl.col("type") == "bridge")
        return f"Bridge questions: {bridge.height}\nSample: {bridge['question'][0]}"
    elif "top" in desc or "longest" in desc:
        df_with_len = df.with_columns(pl.col("text").str.len_chars().alias("text_len"))
        top = df_with_len.sort("text_len", descending=True).head(5)
        return f"Top 5 by text length:\n{top.select('question', 'text_len')}"
    else:
        return f"Dataset has {df.height} rows. Columns: {df.columns}"


def answer_question(question: str) -> str:
    """Extract answer evidence for a question from the corpus text itself.

    Args:
        question: The natural-language question to answer.

    Returns:
        The highest-overlap evidence sentences from the most relevant
        documents, with provenance. This tool NEVER reads the dataset's
        stored answer labels — the agent must synthesise the final answer
        from the returned evidence, exactly like a production RAG tool
        that only sees raw documents.
    """
    df = _require_data()
    q_terms = _content_words(question)
    if not q_terms:
        return "Could not extract content words from that question — rephrase it."

    # Rank documents by content-word overlap with the question.
    scored_docs = []
    for i, row in enumerate(df.iter_rows(named=True)):
        text_l = row["text"].lower()
        doc_score = sum(text_l.count(term) for term in q_terms)
        if doc_score > 0:
            scored_docs.append((doc_score, i, row))
    if not scored_docs:
        return "No documents in the corpus overlap with that question."
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    # Inside each top document, extract the most relevant SENTENCES.
    sections = []
    for doc_score, idx, row in scored_docs[:2]:
        sentences = re.split(r"(?<=[.!?])\s+", row["text"])
        scored_sents = sorted(
            (
                (sum(s.lower().count(term) for term in q_terms), s.strip())
                for s in sentences
                if s.strip()
            ),
            key=lambda x: x[0],
            reverse=True,
        )
        evidence = [s for sc, s in scored_sents[:3] if sc > 0]
        if not evidence:
            continue
        excerpt = "\n".join(f"  - {sent}" for sent in evidence)
        sections.append(f"[Doc {idx}] (overlap={doc_score})\n{excerpt}")

    if not sections:
        return "Relevant documents found but no sentence matched the question terms."
    return (
        "\n\n".join(sections)
        + "\n\n(Evidence extracted from the corpus text. Synthesise the final "
        "answer from these sentences — this tool has no access to answer labels.)"
    )


_STOPWORDS = frozenset(
    "the a an is are was were of in on at to and or for with what which who "
    "whom whose when where why how did does do by as that this it its from "
    "be been has have had not no yes".split()
)


def _content_words(text: str) -> list[str]:
    """Lower-cased, punctuation-stripped question terms minus stopwords."""
    return [
        w
        for w in (t.strip("?,.\"'():;").lower() for t in text.split())
        if w and w not in _STOPWORDS
    ]


def make_tools(qa_data: pl.DataFrame) -> list:
    """Bind the qa_data DataFrame to the tool closures and return the list.

    Args:
        qa_data: The HotpotQA DataFrame from load_hotpotqa().

    Returns:
        List of 4 tool callables. Pass them through build_tool_registry()
        before handing them to a Delegate.
    """
    global _qa_data
    _qa_data = qa_data
    return [data_summary, search_documents, run_query, answer_question]


def tool_schemas(tools: list) -> list[dict]:
    """Build JSON Schema descriptors for a list of tool callables.

    Produces the function-calling shape every tool-capable chat model
    expects (name, description, parameters.properties + required).
    build_tool_registry() registers exactly these schemas with Kaizen.
    """
    schemas = []
    for tool in tools:
        sig = inspect.signature(tool)
        params = {}
        required = []
        for name, param in sig.parameters.items():
            # `from __future__ import annotations` keeps hints as strings.
            annotation = param.annotation
            param_type = "string"
            if annotation in (int, "int"):
                param_type = "integer"
            elif annotation in (float, "float"):
                param_type = "number"
            params[name] = {
                "type": param_type,
                "description": f"Parameter: {name}",
            }
            if param.default is inspect.Parameter.empty:
                required.append(name)
        schemas.append(
            {
                "name": tool.__name__,
                "description": (tool.__doc__ or "").strip().split("\n")[0],
                "parameters": {
                    "type": "object",
                    "properties": params,
                    "required": required,
                },
            }
        )
    return schemas


def build_tool_registry(tools: list[Callable[..., str]]) -> Any:
    """Register plain Python tools on a Kaizen ``ToolRegistry``.

    A Kaizen ``Delegate`` only calls tools that live in its registry: each
    entry is (name, description, JSON-schema parameters, async executor).
    Handing a Delegate a bare list of Python callables registers NOTHING,
    so this helper is the bridge between "functions with docstrings" and
    "tools the model can call".

    Args:
        tools: Callables returning ``str`` (e.g. from :func:`make_tools`).

    Returns:
        A populated ``kaizen_agents.delegate.loop.ToolRegistry``.
    """
    from kaizen_agents.delegate.loop import ToolRegistry

    registry = ToolRegistry()
    for tool, schema in zip(tools, tool_schemas(tools)):

        async def _executor(_fn: Callable[..., str] = tool, **kwargs: Any) -> str:
            return str(_fn(**kwargs))

        registry.register(
            name=schema["name"],
            description=schema["description"],
            parameters=schema["parameters"],
            executor=_executor,
        )
    return registry


def require_llm_trace(trace: Any) -> Any:
    """Raise if a captured Delegate run never produced a real LLM answer.

    ``Delegate.run`` converts exceptions (daemon down, model missing) into an
    ``ErrorEvent`` instead of raising, so a failed run would otherwise look
    like an empty but "successful" result. This check makes it loud.

    Args:
        trace: The ``AgentTrace`` returned by ``obs.agent.capture_run``.

    Returns:
        The same trace, for chaining.
    """
    llm_errors = [ev.error for ev in trace.events if ev.kind == "error" and not ev.tool]
    if llm_errors:
        raise RuntimeError(f"Agent run failed: {llm_errors[0]}. {OLLAMA_HINT}")
    if not trace.filter_kind("complete"):
        raise RuntimeError(f"Agent run produced no completion event. {OLLAMA_HINT}")
    return trace


def require_agent_result(result: dict, agent_name: str) -> dict:
    """Raise if a BaseAgent ``run_async`` call returned an error dict.

    ``BaseAgent.run_async`` reports provider failures as
    ``{"error": ..., "success": False}`` rather than raising.

    Args:
        result: The dict returned by ``await agent.run_async(...)``.
        agent_name: Label used in the error message.

    Returns:
        The same result dict when the call succeeded.
    """
    if not isinstance(result, dict) or "error" in result or result.get("success") is False:
        detail = result.get("error") if isinstance(result, dict) else repr(result)
        raise RuntimeError(f"{agent_name} LLM call failed: {detail}. {OLLAMA_HINT}")
    return result


def print_tool_registry(tools: list) -> None:
    """Human-readable dump of tool names, schemas, and first-line docs."""
    print("Registered tools:")
    for tool in tools:
        doc_first = (tool.__doc__ or "").strip().split("\n")[0]
        print(f"  {tool.__name__}: {doc_first}")
    schemas = tool_schemas(tools)
    print(f"\nGenerated {len(schemas)} JSON Schema descriptors " f"(first shown):")
    print(json.dumps(schemas[0], indent=2)[:400])


__all__ = [
    "MODEL",
    "OLLAMA_HINT",
    "OUTPUT_DIR",
    "load_hotpotqa",
    "make_tools",
    "tool_schemas",
    "build_tool_registry",
    "require_llm_trace",
    "require_agent_result",
    "print_tool_registry",
    "data_summary",
    "search_documents",
    "run_query",
    "answer_question",
]




# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 5.4: Critic Agent — Iterative Refinement
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Compose multiple BaseAgents into a quality-assurance loop
#   - Define a critic Signature that returns should_revise: bool
#   - Build a refinement agent that consumes critic feedback
#   - Run the full Analyse -> Critique -> Refine cycle
#   - Distinguish critic refinement from self-consistency sampling
#
# PREREQUISITES: 03_structured_agent.py (Signature + BaseAgent basics)
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Load data + summary
#   2. Declare Critic and Refined signatures + agents
#   3. Run the Analyse -> Critique -> Refine loop
#   4. Visualise the delta between initial and refined output
#   5. Apply: Singapore regulatory compliance drafting scenario
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

import matplotlib.pyplot as plt

from kaizen import InputField, OutputField, Signature
from kaizen.core.base_agent import BaseAgent



## TASK 1 — Data + summary


In [ ]:
qa_data = load_hotpotqa()
tools = make_tools(qa_data)
summary_text = data_summary()
print(f"Loaded {qa_data.height} examples\n")



### Checkpoint 1


In [ ]:
assert qa_data.height > 0
assert len(summary_text) > 100
print("✓ Checkpoint 1 passed — infra ready\n")



## THEORY — Critic agents vs self-consistency


In [ ]:
# The critic pattern is a QA loop over LLM output:
#
#   Step 1: Analyst agent produces an analysis
#   Step 2: Critic agent reviews it and returns strengths, weaknesses,
#           suggestions, and a should_revise boolean
#   Step 3: If should_revise, a refiner agent produces an improved
#           analysis that incorporates the critic's suggestions
#
# This is NOT the same as self-consistency (MLFP06 Ex 1.5,
# 05_self_consistency.py).
# Self-consistency samples N INDEPENDENT answers and picks the majority.
# The critic REVIEWS a specific answer and suggests targeted fixes.
#
# When to use which:
#   Self-consistency : correctness-sensitive, short answers, parallelisable
#   Critic loop      : long-form analysis, quality-sensitive, can afford
#                      2-3x more LLM calls for substantially better output
#
# ANALOGY: Self-consistency is "ask 5 doctors, take the majority diagnosis."
# Critic loop is "junior doctor writes the diagnosis, senior doctor reviews
# and says what to change."  Both improve quality; they do it differently.



## TASK 2 — Declare signatures and agents


Analyse a dataset summary and produce structured insights.


Critique an analysis and suggest improvements.


Produce an improved analysis incorporating critic feedback.


In [ ]:
#
# We reuse the DataAnalysisSignature structure as the analyst.  Then we
# add a CriticSignature and a RefinedAnalysisSignature.


class DataAnalysisSignature(Signature):

    dataset_summary: str = InputField(description="Statistical summary")
    analysis_question: str = InputField(description="Question to investigate")

    key_findings: list[str] = OutputField(description="Top 3-5 findings")
    recommended_approach: str = OutputField(description="Best ML approach")
    data_quality_issues: list[str] = OutputField(description="Quality concerns")
    next_steps: list[str] = OutputField(description="3-5 next steps")
    confidence: float = OutputField(description="Confidence 0.0-1.0")


class CriticSignature(Signature):

    original_analysis: str = InputField(description="The analysis to critique")
    analysis_question: str = InputField(description="The original question asked")

    strengths: list[str] = OutputField(description="What the analysis does well")
    weaknesses: list[str] = OutputField(description="Gaps or errors in the analysis")
    suggestions: list[str] = OutputField(description="Specific improvement suggestions")
    quality_score: float = OutputField(description="Overall quality 0.0 to 1.0")
    # TODO: Declare should_revise: bool as an OutputField with description
    #       "Whether the analysis needs revision"
    should_revise: bool = ____


class RefinedAnalysisSignature(Signature):

    dataset_summary: str = InputField(description="Dataset summary")
    analysis_question: str = InputField(description="Question to analyse")
    critic_feedback: str = InputField(description="Critic's improvement suggestions")

    improved_findings: list[str] = OutputField(description="Revised findings")
    methodology_note: str = OutputField(description="How the analysis was improved")
    confidence: float = OutputField(description="Confidence after revision (0-1)")


# Domain configs — dataclass pattern (kaizen 2.7.3).  Each agent type
# owns its budget so the critic loop can meter each step independently.


# kaizen 2.28: run_async() requires use_async_llm=True; json_object
# response_format + explicit mode make the typed Signature fields parse out of
# the model reply (otherwise the result is prose, not typed fields).
@dataclass
class DataAnalysisConfig:
    llm_provider: str = "ollama"
    base_url: str = OLLAMA_BASE_URL
    model: str = MODEL  # resolved from OLLAMA_CHAT_MODEL in .env
    temperature: float = 0.2
    budget_limit_usd: float = 1.0
    use_async_llm: bool = True
    response_format: dict = field(default_factory=lambda: {"type": "json_object"})
    structured_output_mode: str = "explicit"


@dataclass
class CriticConfig:
    llm_provider: str = "ollama"
    base_url: str = OLLAMA_BASE_URL
    model: str = MODEL
    temperature: float = 0.2
    # TODO: Set the $1.00 dollar cap field
    budget_limit_usd: float = ____
    use_async_llm: bool = True
    response_format: dict = field(default_factory=lambda: {"type": "json_object"})
    structured_output_mode: str = "explicit"


@dataclass
class RefinedAnalysisConfig:
    llm_provider: str = "ollama"
    base_url: str = OLLAMA_BASE_URL
    model: str = MODEL
    temperature: float = 0.2
    # TODO: Set the $1.00 dollar cap field
    budget_limit_usd: float = ____
    use_async_llm: bool = True
    response_format: dict = field(default_factory=lambda: {"type": "json_object"})
    structured_output_mode: str = "explicit"


class DataAnalysisAgent(BaseAgent):
    def __init__(self, config: DataAnalysisConfig):
        super().__init__(config=config, signature=DataAnalysisSignature())


class CriticAgent(BaseAgent):
    def __init__(self, config: CriticConfig):
        # TODO: Call super().__init__ with the config and a CriticSignature
        #       INSTANCE.
        super().__init__(____)


class RefinedAnalysisAgent(BaseAgent):
    def __init__(self, config: RefinedAnalysisConfig):
        # TODO: Call super().__init__ with the config and a
        #       RefinedAnalysisSignature INSTANCE.
        super().__init__(____)



### Checkpoint 2


In [ ]:
# The class-level `signature = XxxSignature` pattern is a SILENT bug in
# kaizen 2.7.3 — BaseAgent ignores class-level attrs and falls back to
# DefaultSignature.  We check that each agent's INSTANCE signature is
# an instance of its declared Signature subclass, which is the only
# observable proof that the canonical pattern is wired correctly.
_analyst_probe = DataAnalysisAgent(DataAnalysisConfig())
_critic_probe = CriticAgent(CriticConfig())
_refiner_probe = RefinedAnalysisAgent(RefinedAnalysisConfig())
assert isinstance(_analyst_probe.signature, DataAnalysisSignature)
assert isinstance(_critic_probe.signature, CriticSignature)
assert isinstance(_refiner_probe.signature, RefinedAnalysisSignature)
print("✓ Checkpoint 2 passed — 3 signatures + 3 agents declared\n")



## TASK 3 — Run the Analyse -> Critique -> Refine loop


In [ ]:
QUESTION = "What makes multi-hop QA harder than single-hop QA?"


async def iterative_refinement():
    print("Step 1: initial analysis...")
    analyst = DataAnalysisAgent(DataAnalysisConfig())
    # TODO: Await analyst.run_async with dataset_summary=summary_text,
    #       analysis_question=QUESTION, and pass the result through
    #       require_agent_result(..., "analyst").  Note: .run_async(...).
    initial = ____
    initial_text = (
        f"Findings: {initial['key_findings']}\n"
        f"Approach: {initial['recommended_approach']}\n"
        f"Next steps: {initial['next_steps']}"
    )
    print(f"  Initial confidence: {initial['confidence']:.2f}")
    print(f"  Findings: {initial['key_findings'][:2]}")

    print("\nStep 2: critic reviews...")
    critic = CriticAgent(CriticConfig())
    # TODO: Await critic.run_async with original_analysis=initial_text,
    #       analysis_question=QUESTION — checked with require_agent_result
    critique = ____
    print(f"  Quality score: {critique['quality_score']:.2f}")
    print(f"  Should revise: {critique['should_revise']}")
    print(f"  Weaknesses:    {critique['weaknesses'][:2]}")

    if critique["should_revise"]:
        print("\nStep 3: refining based on critic feedback...")
        refiner = RefinedAnalysisAgent(RefinedAnalysisConfig())
        # TODO: Await refiner.run_async with dataset_summary=summary_text,
        #       analysis_question=QUESTION,
        #       critic_feedback=str(critique["suggestions"]) — checked with
        #       require_agent_result
        refined = ____
        print(f"  Refined confidence: {refined['confidence']:.2f}")
        return initial, critique, refined

    print("\nStep 3: critic approves — no revision needed.")
    return initial, critique, None


preflight_ollama(required_models=[MODEL])  # fails loudly if Ollama is down
initial, critique, refined  = await iterative_refinement()



### Checkpoint 3


In [ ]:
assert initial is not None, "Task 3: initial analysis required"
assert critique is not None, "Task 3: critic output required"
assert "should_revise" in critique
assert "quality_score" in critique
assert 0 <= critique["quality_score"] <= 1
print("\n✓ Checkpoint 3 passed — refinement loop executed\n")



## TASK 4 — Visualise the delta between initial and refined


In [ ]:
print("=" * 70)
print("  Refinement Delta")
print("=" * 70)

print(f"\nINITIAL confidence:  {initial['confidence']:.2f}")
print(f"Critic quality:      {critique['quality_score']:.2f}")
print(f"Critic decision:     {'REVISE' if critique['should_revise'] else 'APPROVE'}")
if refined is not None:
    print(f"REFINED confidence:  {refined['confidence']:.2f}")
    delta = refined["confidence"] - initial["confidence"]
    direction = "↑" if delta >= 0 else "↓"
    print(f"Confidence delta:    {direction} {abs(delta):.2f}")
    print(f"\nMethodology note: {refined['methodology_note'][:200]}...")
    print(f"\nImproved findings ({len(refined['improved_findings'])}):")
    for i, f in enumerate(refined["improved_findings"][:5], 1):
        print(f"  {i}. {f}")
else:
    print("(No refinement — critic approved the initial analysis.)")



### Checkpoint 4


In [ ]:
assert isinstance(critique["strengths"], list)
assert isinstance(critique["weaknesses"], list)
assert isinstance(critique["suggestions"], list)
if refined is not None:
    assert isinstance(refined["improved_findings"], list)
    assert 0 <= refined["confidence"] <= 1
print("\n✓ Checkpoint 4 passed — refinement delta visualised\n")

# INTERPRETATION: The critic's quality_score is the first gate.  If the
# score is high and should_revise is False, you save one LLM call and
# ship the initial analysis.  If the score is low and should_revise is
# True, the refiner fires and produces a second-pass result.  The
# confidence delta is your quality signal — if refined.confidence is
# not meaningfully higher than initial.confidence, the critic pattern
# isn't helping on this task shape and you should disable it.



## TASK 5 — Apply: Singapore regulatory compliance drafting


Each critic loop adds 2 LLM calls (a few cents on a hosted model,
  compute only on local Ollama) but can save 1-3 hours of senior human
  review (S$100+ at the rates above).  The pattern is boring to read
  and devastating to deploy.


In [ ]:
# SCENARIO (illustrative figures): A Singapore fintech drafts Monetary Authority of Singapore
# (MAS) compliance memos every time a new product ships.  Memos are
# long-form, high-stakes, and must reference specific MAS notices.
# One bad memo = regulatory finding = S$50K+ fine.
#
# BEFORE: Senior compliance officers (S$150K/year) draft every memo
# personally.  Each memo takes ~3 hours.  400 memos/year * 3h = 1200h
# of senior time = ~S$90,000/year in compliance-officer hours.
#
# NAIVE LLM APPROACH: Hand the product brief to an LLM, get a memo back,
# ship it.  Problem: the LLM hallucinates MAS notice numbers, miscites
# regulations, and the compliance team has to rewrite everything.  Net
# time saved: near zero.  Net risk: high (hallucinated citations).
#
# WITH CRITIC LOOP:
#   1. DraftAgent     writes the first-pass memo
#   2. ComplianceCritic reviews against a checklist (cite MAS notices,
#      reference correct licence tier, flag consumer-protection gaps)
#   3. RefinedDraftAgent produces a second pass incorporating fixes
#   4. Human compliance officer reviews the REFINED draft (not the
#      raw first pass) — 20 minutes instead of 3 hours
#
# BUSINESS IMPACT:
#   - Senior time per memo: 3h -> 20 min  (9x reduction)
#   - Annual hours saved:   1200h -> 133h = 1067h ~ S$80,000/year
#   - Quality floor:        the critic enforces the checklist; first
#                           drafts that miss items never reach the
#                           human reviewer
#   - Audit trail:          critic JSON output persists as evidence
#                           that the checklist was applied — this is
#                           itself a regulatory asset
#
# THE GENERAL PATTERN: Critic loops shine wherever the cost of a human
# review is high and the cost of two extra LLM calls is negligible.


print("\n" + "=" * 70)
print("  KEY TAKEAWAY: Critic loops trade LLM spend for human time")
print("=" * 70)
print(
)



## VISUALISATION — Quality improvement: initial vs refined


In [ ]:
stages = ["Initial\nAnalysis", "Critic\nScore", "Refined\nAnalysis"]
scores = [
    initial["confidence"],
    critique["quality_score"],
    refined["confidence"] if refined is not None else initial["confidence"],
]
colors = ["#90CAF9", "#FFE082", "#A5D6A7"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(stages, scores, color=colors, edgecolor="#333", linewidth=0.8)
for bar, score in zip(bars, scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{score:.2f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
ax.set_ylim(0, 1.15)
ax.set_ylabel("Confidence / Quality Score")
ax.set_title("Critic Loop: Quality Improvement Across Stages")
ax.axhline(y=0.8, color="green", linestyle="--", alpha=0.5, label="Quality threshold")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_critic_improvement.png", dpi=150)
plt.close(fig)
print(f"\nSaved: {OUTPUT_DIR / '04_critic_improvement.png'}")



## DIAGNOSTIC CHECKPOINT


In [ ]:
# The Agent Trace lens needs a tool-using Delegate (01 and 02).  Here
# the diagnostic is the loop itself: the critic's quality_score and
# should_revise decision, and the confidence delta plotted above.  If
# refined confidence is not meaningfully above the initial confidence
# across several questions, the critic loop is not paying for itself.



## REFLECTION


[x] Composed 3 BaseAgents (analyst, critic, refiner) in a loop
  [x] Declared should_revise as a typed bool that gates the refiner
  [x] Ran the Analyse -> Critique -> Refine cycle end-to-end
  [x] Measured the confidence delta as a quality signal
  [x] Distinguished critic refinement from self-consistency sampling

  KEY INSIGHT: Three agents doing one job, each with a single
  responsibility: analyse, critique, refine.  This is the same pattern
  as a code-review workflow — a junior writes, a senior reviews, the
  junior revises.  You just automated both halves of it.

  MODULE WRAPUP: Exercise 5 covered four agent patterns — ReAct for
  tool exploration, budgeted agents for cost safety, structured agents
  for pipeline integration, and critic loops for quality assurance.
  Together they are the production agent toolkit.  Exercise 6
  (Multi-Agent Systems) composes specialists with a supervisor for
  fan-out / fan-in orchestration.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

